# Cross-domain transfer: DALES-pretrained -> Toronto-3D

## Reading and visualizing raw point clouds using `Data` objects

In [ ]:
import os
import sys

# Add the project's files to the python path
file_path = os.path.dirname(os.path.abspath(''))  # for .ipynb notebook
sys.path.append(file_path)

import numpy as np
import torch
from plyfile import PlyData
from src.data import Data
from src.utils.color import to_float_rgb


# ============================================================
# Toronto-3D constants
# ============================================================

TORONTO3D_NUM_CLASSES = 8

# Toronto-3D original labels:
#   0 = Unclassified
#   1 = Road
#   2 = Road marking
#   3 = Natural
#   4 = Building
#   5 = Utility line
#   6 = Pole
#   7 = Car
#   8 = Fence

# Mapping: original label -> train ID
# We put unclassified into "Ignored" (= TORONTO3D_NUM_CLASSES = 8)
TORONTO3D_ID2TRAINID = np.asarray([
    TORONTO3D_NUM_CLASSES,  # 0 Unclassified   -> 8 Ignored
    0,                      # 1 Road            -> 0 Road
    1,                      # 2 Road marking    -> 1 Road marking
    2,                      # 3 Natural         -> 2 Natural
    3,                      # 4 Building        -> 3 Building
    4,                      # 5 Utility line    -> 4 Utility line
    5,                      # 6 Pole            -> 5 Pole
    6,                      # 7 Car             -> 6 Car
    7,                      # 8 Fence           -> 7 Fence
])

TORONTO3D_CLASS_NAMES = [
    'Road',
    'Road marking',
    'Natural',
    'Building',
    'Utility line',
    'Pole',
    'Car',
    'Fence',
    'Ignored',
]

TORONTO3D_CLASS_COLORS = np.asarray([
    [128, 128, 128],  # Road - grey
    [255, 255,   0],  # Road marking - yellow
    [ 70, 115,  66],  # Natural - green
    [214,  66,  54],  # Building - red
    [255, 165,   0],  # Utility line - orange
    [186, 160, 164],  # Pole - mauve
    [  0, 128, 255],  # Car - blue
    [160,  82,  45],  # Fence - brown
    [  0,   0,   0],  # Ignored - black
])


# ============================================================
# Mapping Toronto-3D -> DALES (for cross-domain inference)
# ============================================================
# DALES classes: 0=ground, 1=vegetation, 2=cars, 3=trucks,
#                4=powerlines, 5=fences, 6=poles, 7=buildings
# Toronto-3D (train IDs after remap above):
#   0=Road, 1=Road marking, 2=Natural, 3=Building,
#   4=Utility line, 5=Pole, 6=Car, 7=Fence

DALES_NUM_CLASSES = 8

TORONTO3D_TO_DALES = np.asarray([
    0,                  # Road         -> 0 Ground
    DALES_NUM_CLASSES,  # Road marking -> Ignored (no DALES equivalent)
    1,                  # Natural      -> 1 Vegetation
    7,                  # Building     -> 7 Buildings
    4,                  # Utility line -> 4 Powerlines
    6,                  # Pole         -> 6 Poles
    2,                  # Car          -> 2 Cars
    5,                  # Fence        -> 5 Fences
    DALES_NUM_CLASSES,  # Ignored      -> Ignored
])


# ============================================================
# Reader function
# ============================================================

def read_toronto3d_tile(
        filepath,
        xyz=True,
        rgb=True,
        intensity=True,
        semantic=True,
        instance=False,
        remap=True,
        max_intensity=600):
    """Read a Toronto-3D tile saved as PLY.

    :param filepath: str
        Absolute path to the PLY file (e.g. L001.ply)
    :param xyz: bool
        Whether XYZ coordinates should be saved in Data.pos
    :param rgb: bool
        Whether RGB colors should be saved in Data.rgb
    :param intensity: bool
        Whether intensity should be saved in Data.intensity
    :param semantic: bool
        Whether semantic labels should be saved in Data.y
    :param instance: bool
        Not supported for Toronto-3D
    :param remap: bool
        Whether semantic labels should be mapped to train IDs
    :param max_intensity: float
        Maximum value to clip intensity before normalizing to [0, 1]
    """
    data = Data()

    # Read the PLY file
    plydata = PlyData.read(filepath)
    vertex = plydata['vertex']

    # Populate coordinates
    if xyz:
        pos = torch.stack([
            torch.FloatTensor(np.array(vertex['x'])),
            torch.FloatTensor(np.array(vertex['y'])),
            torch.FloatTensor(np.array(vertex['z'])),
        ], dim=-1)

        # Toronto-3D uses UTM coordinates with very large values.
        # Subtract the first point to avoid float32 precision issues.
        pos_offset = pos[0].double()
        data.pos = (pos.double() - pos_offset).float()
        data.pos_offset = pos_offset

    # Populate RGB (Toronto-3D stores RGB in [0, 255])
    if rgb:
        data.rgb = to_float_rgb(torch.stack([
            torch.FloatTensor(np.array(vertex['red']).astype('float32') / 255.0),
            torch.FloatTensor(np.array(vertex['green']).astype('float32') / 255.0),
            torch.FloatTensor(np.array(vertex['blue']).astype('float32') / 255.0),
        ], dim=-1))

    # Populate intensity
    if intensity:
        data.intensity = torch.FloatTensor(
            np.array(vertex['scalar_Intensity']).astype('float32')
        ).clip(min=0, max=max_intensity) / max_intensity

    # Populate semantic labels
    if semantic:
        y = torch.LongTensor(np.array(vertex['scalar_Label']).astype('int64'))
        data.y = torch.from_numpy(TORONTO3D_ID2TRAINID)[y] if remap else y

    if instance:
        raise NotImplementedError(
            "Toronto-3D does not contain instance labels.")

    return data

### `Data` visualization

In [3]:
filepath = '/Data/aziz.bacha/datasets/Toronto_3D/L002.ply'
data = read_toronto3d_tile(filepath)

In [4]:
data

Data(pos=[10283800, 3], pos_offset=[3], rgb=[10283800, 3], intensity=[10283800], y=[10283800])

In [5]:
data.num_points

10283800

In [6]:
data.keys

['pos', 'rgb', 'intensity', 'pos_offset', 'y']

In [ ]:
data.show(class_names=TORONTO3D_CLASS_NAMES, class_colors=TORONTO3D_CLASS_COLORS)

In [ ]:
data.show(center=data.pos.mean(dim=0), radius=50, keys=['intensity'], class_names=TORONTO3D_CLASS_NAMES, class_colors=TORONTO3D_CLASS_COLORS)

## Tiling

In [ ]:
from src.transforms import SampleXYTiling, GridSampling3D
from src.data import Batch

# Tile the cloud into `xy_tiling` XY-oriented chunks of equal horizontal 
# span
xy_tiling = (4, 2)

# Voxelize the point cloud only for the sake of faster computation and 
# visualization here
data_5m = GridSampling3D(10)(data)

# Compute each chunk 
chunks = []
for x in range(xy_tiling[0]):
    for y in range(xy_tiling[1]):        
        # Extract the chunk at (x, y) in the tiling grid
        chunk = SampleXYTiling(x=x, y=y, tiling=xy_tiling)(data_5m)

        # Add a 'tile' attribute to the points for visualization
        chunk.tile = torch.full((chunk.num_points,), x * xy_tiling[1] + y)
        
        # Store the chunk for later aggregation
        chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

# Show the resulting `Data' with the 'tile' attribute
data_tiled.show(keys='tile')

In [ ]:
from src.transforms import SampleRecursiveMainXYAxisTiling, GridSampling3D
from src.data import Batch

# Recursively tile the cloud into `2**pc_tiling` chunks with respect to 
# principal components of the XY coordiantes
pc_tiling = 3

# Voxelize the point cloud only for the sake of faster computation and 
# visualization here
data_5m = GridSampling3D(5)(data)

# Compute each chunk 
chunks = []
for x in range(2**pc_tiling):
    # Extract the chunk at x in the recursive tiling
    chunk = SampleRecursiveMainXYAxisTiling(x=x, steps=pc_tiling)(data_5m)

    # Add a 'tile' attribute to the points for visualization
    chunk.tile = torch.full((chunk.num_points,), x)
    
    # Store the chunk for later aggregation
    chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

# Show the resulting `Data' with the 'tile' attribute
data_tiled.show(keys='tile')

In [ ]:
from src.transforms import SampleXYTiling

# Extract the chunk at (x, y) in the tiling grid
data = SampleXYTiling(x=1, y=1, tiling=3)(data)

## Using a pretrained model for inference

### Instantiating transforms from `configs/`

In [7]:
from src.utils import init_config

cfg = init_config(overrides=[f"experiment=semantic/dales"])

In [8]:
cfg.keys()

dict_keys(['task_name', 'optimized_metric', 'tags', 'train', 'test', 'compile', 'ckpt_path', 'seed', 'float32_matmul_precision', 'datamodule', 'model', 'callbacks', 'logger', 'trainer', 'paths', 'extras'])

In [9]:
from src.transforms import instantiate_datamodule_transforms

transforms_dict = instantiate_datamodule_transforms(cfg.datamodule)
transforms_dict

{'pre_transform': Compose([
   DataTo(device=cuda),
   SaveNodeIndex(key=sub),
   GridSampling3D(grid_size=0.1, quantize_coords=False, allow_negative_coords=False, mode=mean, bins={'y': 9}, chunk_size=10000000),
   KNN(k=25, r_max=10, oversample=False, self_is_neighbor=False, save_as_csr=False),
   PointFeatures(keys=('elevation', 'intensity', 'linearity', 'planarity', 'scattering', 'verticality'), k_min=1, k_step=-1, k_min_search=10, add_self_as_neighbor=True, chunk_size=100000, overwrite=True),
   GroundElevation(z_threshold=5, verticality_threshold=None, xy_grid=5, model=ransac, scale=20, kwargs={}),
   AdjacencyGraph(k=10, w=1),
   ConnectIsolated(k=1),
   AddKeysTo(keys=['linearity', 'planarity', 'scattering', 'elevation'], to=x, delete_after=False),
   CutPursuitPartition(regularization=[0.1, 0.2, 0.3], spatial_weight=[0.1, 0.01, 0.001], cutoff=[10, 30, 100], iterations=15, k_adjacency=10, edge_reduce=mean),
   NAGRemoveKeys(level=all, keys=('x',)),
   SegmentFeatures(n_max=128, 

The transforms are chained operations applied to a `Data` or a `NAG` object. Their order and parametrization plays a significant role and modifying these may have non-negligible downstream effects. **These must be thought as part of the model itself**.

### Applying transforms

In [10]:
# Re-read your data (since 'data' got overwritten)
filepath = '/Data/aziz.bacha/datasets/Toronto_3D/L002.ply'
input_data = read_toronto3d_tile(filepath)

# If you were filtering unclassified points, do it again
mask = input_data.y < TORONTO3D_NUM_CLASSES
from src.data import Data
clean = Data()
clean.pos = input_data.pos[mask]
clean.rgb = input_data.rgb[mask]
clean.intensity = input_data.intensity[mask]
clean.y = input_data.y[mask]
clean.pos_offset = input_data.pos_offset
input_data = clean

# DON'T do "from src.transforms import *" — import only what you need
from src.transforms import GroundElevation

# Now run transforms one by one
transforms = transforms_dict['pre_transform'].transforms

for i, t in enumerate(transforms):
    name = t.__class__.__name__
    print(f"{i} Applying {name}...")

    if name == 'GroundElevation':
        input_data.elevation = input_data.pos[:, 2] - input_data.pos[:, 2].min()
        print("  -> Replaced with z-based elevation")
        continue

    input_data = t(input_data)
    print(f"  -> Output type: {type(input_data).__name__}")

nag = input_data

0 Applying DataTo...
  -> Output type: Data
1 Applying SaveNodeIndex...
  -> Output type: Data
2 Applying GridSampling3D...
  -> Output type: Data
3 Applying KNN...
  -> Output type: Data
4 Applying PointFeatures...
  -> Output type: Data
5 Applying GroundElevation...
  -> Replaced with z-based elevation
6 Applying AdjacencyGraph...
  -> Output type: Data
7 Applying ConnectIsolated...
  -> Output type: Data
8 Applying AddKeysTo...
  -> Output type: Data
9 Applying CutPursuitPartition...
  -> Output type: NAG
10 Applying NAGRemoveKeys...
  -> Output type: NAG
11 Applying SegmentFeatures...
  -> Output type: NAG
12 Applying RadiusHorizontalGraph...
  -> Output type: NAG
13 Applying NAGTo...
  -> Output type: NAG


In [ ]:
# Simulate the behavior of the dataset's I/O behavior with only
# `point_load_keys` and `segment_load_keys` loaded from disk
from src.transforms import NAGRemoveKeys
nag = NAGRemoveKeys(level=0, keys=[k for k in nag[0].keys if k not in cfg.datamodule.point_load_keys])(nag)
nag = NAGRemoveKeys(level='1+', keys=[k for k in nag[1].keys if k not in cfg.datamodule.segment_load_keys])(nag)

# Move to device
nag = nag.cuda()

# Apply on-device transforms
nag = transforms_dict['on_device_test_transform'](nag)

In [12]:
nag

NAG(num_levels=4, num_points=[1164622, 49606, 15345, 3699], device=cuda:0, ratios={'|P_0| / |P_1|': '23.5', '|P_1| / |P_2|': '3.2', '|P_2| / |P_3|': '4.1'})

Let's visualize the impact of the transforms on the data on a small area for high-resolution display. Note we can display the nodes and edges of the superpoint graphs by passing `show(centroids=True, h_edge=True)`.

In [ ]:
nag.show(class_names=TORONTO3D_CLASS_NAMES, class_colors=TORONTO3D_CLASS_COLORS, center=nag[0].pos.mean(dim=0).tolist(), radius=20, keys=nag[0].keys, centroids=True, h_edge=True)

### Instantiating a pretrained model from `configs/` and a `*.ckpt`

In [14]:
import hydra 
from src.utils import init_config

# Path to the checkpoint file downloaded from https://zenodo.org/records/8042712
ckpt_path = "/users/eleves-b/2022/aziz.bacha/superpoint_transformer/models/spt-2_dales.ckpt"

cfg = init_config(overrides=[f"experiment=semantic/dales"])

# Instantiate the model and load pretrained weights
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(ckpt_path)

Lightning automatically upgraded your loaded checkpoint from v1.8.4.post0 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../models/spt-2_dales.ckpt`
⚠️ No `__version__` found in checkpoint metadata.
This means the checkpoint was saved with a version of the code prior to 3.0.0.
Setting the version 2.1.0, so that the official weights of SPT and SPC are compatible.
If you have weights from version 2.2.0, please use the migration script to set the version to 3.0.0.
(see: src/utils/backwards_compatibility/add_version_to_checkpoint.py and CHANGELOG.md for more details)
⚠️ No `commit_hash` found in checkpoint metadata.
This means the checkpoint was saved with a version of the code prior to 3.0.0.
Setting the commit hash to unknown.


### 4.4. Applying SPT

In [15]:
# BEFORE running model inference, cap the ground truth labels
# so they don't exceed DALES_NUM_CLASSES - 1
# This prevents CUDA assertions in the model's internal operations
nag[0].y = nag[0].y.clamp(max=DALES_NUM_CLASSES - 1)
for i in range(1, nag.num_levels):
    if hasattr(nag[i], 'y') and nag[i].y is not None:
        nag[i].y = nag[i].y.clamp(max=DALES_NUM_CLASSES - 1)

# Now run inference
model = model.eval().to(nag.device)
with torch.no_grad():
    output = model(nag)

# Then get predictions
pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

In [16]:
output.semantic_pred().shape, nag.num_points

(torch.Size([49606]), [1164622, 49606, 15345, 3699])

In [17]:
# Compute the level-0 (voxel-wise) semantic segmentation predictions 
# based on the predictions on level-1 superpoints and save those for 
# visualization in the level-0 Data under the 'semantic_pred' attribute
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

In [ ]:
from src.datasets.dales import CLASS_NAMES as DALES_CLASS_NAMES
from src.datasets.dales import CLASS_COLORS as DALES_CLASS_COLORS

nag.show(class_names=DALES_CLASS_NAMES, class_colors=DALES_CLASS_COLORS, center=nag[0].pos.mean(dim=0).tolist(), radius=20)

In [19]:
print(f"P0 num points: {nag[0].num_points}")
print(f"P1 num points: {nag[1].num_points}")
print(f"super_index shape: {nag[0].super_index.shape}")
print(f"semantic_pred shape: {output.semantic_pred().shape}")

P0 num points: 1164622
P1 num points: 49606
super_index shape: torch.Size([1164622])
semantic_pred shape: torch.Size([49606])


## IOU per class

In [ ]:
pred_sp = output.semantic_pred().cpu()
pred = pred_sp[nag[0].super_index.cpu()]

gt_raw = nag[0].y.cpu()
print(f"gt_raw shape: {gt_raw.shape}")

if gt_raw.dim() == 2:
    gt_toronto = gt_raw.argmax(dim=1)
else:
    gt_toronto = gt_raw

gt_toronto = gt_toronto.clamp(max=TORONTO3D_NUM_CLASSES)
gt_dales = torch.from_numpy(TORONTO3D_TO_DALES)[gt_toronto]

valid_mask = gt_dales < DALES_NUM_CLASSES
pred_valid = pred[valid_mask]
gt_valid = gt_dales[valid_mask]

print(f"pred_valid: {pred_valid.shape}, gt_valid: {gt_valid.shape}")

# Compute per-class IoU
from src.datasets.dales import CLASS_NAMES as DALES_CLASS_NAMES

ious = []
print(f"\n{'Class':<15} {'IoU':>8} {'Support':>10}")
print("-" * 35)

for c in range(DALES_NUM_CLASSES):
    pred_c = (pred_valid == c)
    gt_c = (gt_valid == c)
    intersection = (pred_c & gt_c).sum().item()
    union = (pred_c | gt_c).sum().item()

    if union == 0:
        print(f"{DALES_CLASS_NAMES[c]:<15} {'N/A':>8} {0:>10}")
        continue

    iou = intersection / union
    ious.append(iou)
    print(f"{DALES_CLASS_NAMES[c]:<15} {iou:>8.4f} {gt_c.sum().item():>10}")

miou = np.mean(ious)
print("-" * 35)
print(f"{'mIoU':<15} {miou:>8.4f}")

gt_raw shape: torch.Size([1164622, 9])
pred_valid: torch.Size([1160149]), gt_valid: torch.Size([1160149])

Class                IoU    Support
-----------------------------------
Ground            0.7590     261465
Vegetation        0.0863     557116
Cars              0.0000      59884
Trucks            0.0000          0
Power lines       0.2050      19183
Fences            0.0207       8609
Poles             0.0080      26678
Buildings         0.2379     227214
-----------------------------------
mIoU              0.1646


In [ ]:
# Oracle mIoU : how good is the partition itself?
# Use DALES-mapped labels
nag[0].y_dales = gt_dales
nag[1].y_dales = nag[0].y_dales  

# print the Toronto-3D oracle
gt_hist = nag[0].y.cpu()  # already a histogram
print("Oracle P1 (Toronto classes):", nag[1].semantic_segmentation_oracle(TORONTO3D_NUM_CLASSES))

Oracle P1 (Toronto classes): {'oa': tensor(93.1883), 'macc': tensor(76.4718), 'miou': tensor(71.8016), 'iou_per_class': tensor([79.7586,  0.2006, 97.2223, 94.5282, 77.0291, 72.5888, 75.3323, 77.7529]), 'seen_class': tensor([True, True, True, True, True, True, True, True])}


In [23]:
print(f"pred shape: {pred.shape}, dtype: {pred.dtype}, device: {pred.device}")
print(f"gt_dales shape: {gt_dales.shape}, dtype: {gt_dales.dtype}, device: {gt_dales.device}")
print(f"valid_mask shape: {valid_mask.shape}, dtype: {valid_mask.dtype}, device: {valid_mask.device}")
print(f"valid_mask sum: {valid_mask.sum()}")

pred shape: torch.Size([1164622]), dtype: torch.int64, device: cuda:0
gt_dales shape: torch.Size([1164622, 9]), dtype: torch.int64, device: cuda:0
valid_mask shape: torch.Size([1164622, 9]), dtype: torch.bool, device: cuda:0
valid_mask sum: 10007827
